## Import Packages

In [40]:
!pip install dask

In [41]:
import numpy as np
import pandas as pd
import dask.dataframe as dd
import matplotlib.pyplot as plt
import seaborn as sns

## Load Data

In [42]:
# paths for the three dfs

df_jan_path = "/kaggle/input/datasets/elemento/nyc-yellow-taxi-trip-data/yellow_tripdata_2016-01.csv"
df_feb_path = "/kaggle/input/datasets/elemento/nyc-yellow-taxi-trip-data/yellow_tripdata_2016-02.csv"
df_mar_path = "/kaggle/input/datasets/elemento/nyc-yellow-taxi-trip-data/yellow_tripdata_2016-03.csv"


- Based on the exploratory data analysis (EDA) of NYC Yellow Taxi trips, we identified outliers in the data. I selected specific columns that are useful for demand prediction, and these columns also contain outliers. Therefore, I need to remove the outliers from these specific columns.
- Columns are 'trip_distance', 'pickup_longitude',
       'pickup_latitude','dropoff_longitude', 'dropoff_latitude', 'fare_amount'

In [43]:
# load the dataframes

df_jan = dd.read_csv(df_jan_path, assume_missing=True, usecols= ['trip_distance', 'tpep_pickup_datetime', 'pickup_longitude',
       'pickup_latitude','dropoff_longitude', 'dropoff_latitude', 'fare_amount'], parse_dates=["tpep_pickup_datetime"])

df_feb = dd.read_csv(df_feb_path, assume_missing=True, usecols= ['trip_distance', 'tpep_pickup_datetime', 'pickup_longitude',
       'pickup_latitude','dropoff_longitude', 'dropoff_latitude', 'fare_amount'], parse_dates=["tpep_pickup_datetime"])


df_mar = dd.read_csv(df_mar_path, assume_missing=True, usecols= ['trip_distance', 'tpep_pickup_datetime', 'pickup_longitude',
       'pickup_latitude','dropoff_longitude', 'dropoff_latitude', 'fare_amount'], parse_dates=["tpep_pickup_datetime"])

In [44]:
# concat the three dataframes as one

df_final = dd.concat([df_jan, df_feb, df_mar], axis=0) # axis =0 means It works vertically (up and down).

In [45]:
df_final.head()

,tpep_pickup_datetime,trip_distance,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,fare_amount
0,2016-01-01,1.10,-73.990372,40.734695,-73.981842,40.732407,7.5
1,2016-01-01,4.90,-73.980782,40.729912,-73.944473,40.716679,18.0
2,2016-01-01,10.54,-73.984550,40.679565,-73.950272,40.788925,33.0
3,2016-01-01,4.75,-73.993469,40.718990,-73.962242,40.657333,16.5
4,2016-01-01,1.76,-73.960625,40.781330,-73.977264,40.758514,8.0


### **New york bounding box:**
min_latitude = 40.60
max_latitude = 40.85
min_longitude = -74.05
max_longitude = -73.70

In [46]:
# set the values of coordinates

min_latitude = 40.60
max_latitude = 40.85
min_longitude = -74.05
max_longitude = -73.70

In [47]:
# fare amount column
fare_amount = df_final["fare_amount"].compute()

# trip distance column
trip_distance = df_final["trip_distance"].compute()

In [48]:
fare_amount.shape[0]/10000000

3.4499859

In [49]:
## Percentile of fare amount
percentiles=np.arange(0.991,1,0.001)
fare_amount.quantile(percentiles)

0.991        52.00
0.992        52.00
0.993        52.00
0.994        52.00
0.995        54.00
0.996        58.50
0.997        63.00
0.998        69.00
0.999        81.00
1.000    429496.72
Name: fare_amount, dtype: float64

In [50]:
max_fare_amount_val = fare_amount.quantile(percentiles).iloc[-2].item()
min_fare_amount_val = 0.50

print(min_fare_amount_val)
print(max_fare_amount_val)

0.5
81.0


In [51]:
trip_distance.quantile(percentiles)

0.991          18.80
0.992          19.00
0.993          19.30
0.994          19.63
0.995          20.04
0.996          20.51
0.997          21.10
0.998          21.90
0.999          24.43
1.000    19072628.80
Name: trip_distance, dtype: float64

In [52]:
# percentile values for trip_distance

min_trip_distance_val = 0.25
max_trip_distance_val = trip_distance.quantile(percentiles).iloc[-2].item()

print(min_trip_distance_val)
print(max_trip_distance_val)

0.25
24.43


## Remove Outlier from the location data

In [53]:
# select data points within the given ranges

df_final = df_final.loc[(df_final["pickup_latitude"].between(min_latitude, max_latitude, inclusive="both")) & 
(df_final["pickup_longitude"].between(min_longitude, max_longitude, inclusive="both")) & 
(df_final["dropoff_latitude"].between(min_latitude, max_latitude, inclusive="both")) & 
(df_final["dropoff_longitude"].between(min_longitude, max_longitude, inclusive="both")), :]

## Remove Outliers from the Fare Amount data and Distance

In [54]:
df_final = df_final.loc[(df_final["fare_amount"].between(min_fare_amount_val,max_fare_amount_val,inclusive="both")) & 
(df_final["trip_distance"].between(min_trip_distance_val,max_trip_distance_val,inclusive="both"))]

## Remove Outliers from the Distance data

In [55]:
df_final = df_final.loc[(df_final["fare_amount"].between(min_fare_amount_val,max_fare_amount_val,inclusive="both")) & 
(df_final["trip_distance"].between(min_trip_distance_val,max_trip_distance_val,inclusive="both"))]

## Save After Removing Outlier

In [56]:
# save the pickup coordinates dataset
import os

dir_path = "/kaggle/working/data/interim/"
file_path = os.path.join(dir_path, "location_data.csv")
pickup_coord_dataset = df_final.loc[:,['tpep_pickup_datetime','pickup_latitude','pickup_longitude']]

In [57]:
# form the dataset

pickup_coord_dataset = df_final.loc[:,['tpep_pickup_datetime','pickup_latitude','pickup_longitude']].compute()

print("Shape of the dataset is ", pickup_coord_dataset.shape)

Shape of the dataset is  (33234199, 3)


In [58]:
import os
os.makedirs(dir_path, exist_ok=True)

In [59]:
pickup_coord_dataset.to_csv(file_path, index=False)

## Making The Regions 

In [86]:
import pandas as pd
from sklearn.cluster import MiniBatchKMeans
from sklearn.preprocessing import StandardScaler

In [87]:
df_reader=pd.read_csv("/kaggle/working/data/interim/location_data.csv",chunksize=100000, usecols=["pickup_latitude","pickup_longitude"])

In [88]:
# train the standard scaler

scaler = StandardScaler()

for chunk in df_reader:
    # fit the scaler
    scaler.partial_fit(chunk)

In [89]:
# 1. Re-initialize the iterator (since cell 73 consumed it completely)
df_reader = pd.read_csv("/kaggle/working/data/interim/location_data.csv", chunksize=100000, usecols=["pickup_latitude", "pickup_longitude"])

# 2. Train the MiniBatchKMeans model
mini_batch = MiniBatchKMeans(n_clusters=30, n_init=10, random_state=42)
for chunk in df_reader:
    scaled_chunk = scaler.transform(chunk)
    mini_batch.partial_fit(scaled_chunk)

# 3. Access centroids
mini_batch.cluster_centers_

array([[ 1.94149572,  0.6614905 ],
       [-0.13980742, -0.07596928],
       [-1.99721801,  1.44528196],
       [-3.83635934,  5.15118517],
       [-1.16343866, -0.81227839],
       [ 0.42137614, -0.13488625],
       [ 0.72209007,  2.8610836 ],
       [-0.52924948, -0.39015448],
       [ 1.07003845,  0.56520318],
       [-2.24451716, -0.3303071 ],
       [-1.00166487, -0.4036074 ],
       [ 1.16185672, -0.10175947],
       [-0.08902033, -0.55920139],
       [ 0.31820801,  1.59046972],
       [-0.04552948, -0.25372558],
       [ 0.20846098, -0.35389222],
       [-1.31780812,  0.52314378],
       [ 2.79102906,  0.81932304],
       [ 0.28712745,  0.12935608],
       [ 0.73099844, -0.27639436],
       [-1.53772972, -1.0082658 ],
       [ 0.67692302,  0.39844824],
       [-0.76412025, -0.74701446],
       [ 1.62394033,  0.1378545 ],
       [-3.07963657, -0.43385234],
       [ 0.3506753 , -0.54018812],
       [-0.57100818, -0.17842182],
       [-0.38830582, -0.76072368],
       [-2.66479788,

In [90]:
scaler.inverse_transform(mini_batch.cluster_centers_)

array([[ 40.80392392, -73.94975046],
       [ 40.74726528, -73.97685385],
       [ 40.69670159, -73.92094427],
       [ 40.64663525, -73.78474354],
       [ 40.7193993 , -74.00391496],
       [ 40.7625422 , -73.97901919],
       [ 40.77072843, -73.8689102 ],
       [ 40.73666362, -73.9884009 ],
       [ 40.78020052, -73.95328925],
       [ 40.68996945, -73.98620137],
       [ 40.72380321, -73.98889533],
       [ 40.78270006, -73.9778017 ],
       [ 40.74864784, -73.99461378],
       [ 40.75973368, -73.91560827],
       [ 40.74983178, -73.98338682],
       [ 40.75674608, -73.98706818],
       [ 40.71519695, -73.95483503],
       [ 40.82705049, -73.94394974],
       [ 40.75888759, -73.96930766],
       [ 40.77097094, -73.98421995],
       [ 40.70921009, -74.01111796],
       [ 40.76949887, -73.95941789],
       [ 40.73026981, -74.00151635],
       [ 40.79527921, -73.96899532],
       [ 40.66723527, -73.9900069 ],
       [ 40.76061753, -73.993915  ],
       [ 40.73552684, -73.98061923],
 

In [91]:
final_df=pd.read_csv("/kaggle/working/data/interim/location_data.csv")

In [92]:
final_df.head()

,tpep_pickup_datetime,pickup_latitude,pickup_longitude
0,2016-01-01 00:00:00,40.734695,-73.990372
1,2016-01-01 00:00:00,40.729912,-73.980782
2,2016-01-01 00:00:00,40.679565,-73.984550
3,2016-01-01 00:00:00,40.718990,-73.993469
4,2016-01-01 00:00:00,40.781330,-73.960625


In [93]:
# prediction 
scaled_location_subset = scaler.transform(final_df.iloc[:, 1:])


scaled_location_subset

# get the cluster predictions

cluster_predictions = mini_batch.predict(scaled_location_subset)

cluster_predictions.shape

(33234199,)

In [103]:
# save the cluster predictions in data

# Save the cluster predictions directly into your Pandas DataFrame
final_df['region'] = cluster_predictions
time_series_data = final_df.drop(columns=["pickup_latitude","pickup_longitude"])

save_path = "/kaggle/working/data/interim/time_series.csv"

time_series_data.to_csv(save_path, index=False)

In [77]:
# import shutil 

# shutil.rmtree("/kaggle/working/data/interim/time_series.csv")

* Time Series Data

In [124]:
time_series_data=pd.read_csv("/kaggle/working/data/interim/time_series.csv")

In [128]:
time_series_data.head()

,tpep_pickup_datetime,region
0,2016-01-01 00:00:00,7
1,2016-01-01 00:00:00,26
2,2016-01-01 00:00:00,9
3,2016-01-01 00:00:00,10
4,2016-01-01 00:00:00,8


In [129]:
time_series_data['tpep_pickup_datetime'] = pd.to_datetime(time_series_data['tpep_pickup_datetime'])

In [130]:
time_series_data.set_index('tpep_pickup_datetime', inplace=True)

time_series_data

,region
tpep_pickup_datetime,
2016-01-01 00:00:00,7
2016-01-01 00:00:00,26
2016-01-01 00:00:00,9
2016-01-01 00:00:00,10
2016-01-01 00:00:00,8
...,...
2016-03-31 21:43:11,3
2016-03-20 08:45:16,3
2016-03-20 08:59:21,3


In [132]:
region_group=time_series_data.groupby("region")

In [133]:
time_series_data.isna().sum()

region    0
dtype: int64

In [134]:
# resample the time series in 15 minute intervals

resampled_data = (
    region_group['region']
    .resample("15min")
    .count()
)

resampled_data

region  tpep_pickup_datetime
0       2016-01-01 00:00:00      58
        2016-01-01 00:15:00     120
        2016-01-01 00:30:00     149
        2016-01-01 00:45:00     160
        2016-01-01 01:00:00     187
                               ... 
29      2016-03-31 22:45:00      14
        2016-03-31 23:00:00      17
        2016-03-31 23:15:00      18
        2016-03-31 23:30:00      13
        2016-03-31 23:45:00      14
Name: region, Length: 262080, dtype: int64

In [135]:

resampled_data.name = "total_pickups"

In [136]:
resampled_data = resampled_data.reset_index(level=0)

resampled_data

,region,total_pickups
tpep_pickup_datetime,,
2016-01-01 00:00:00,0,58
2016-01-01 00:15:00,0,120
2016-01-01 00:30:00,0,149
2016-01-01 00:45:00,0,160
2016-01-01 01:00:00,0,187
...,...,...
2016-03-31 22:45:00,29,14
2016-03-31 23:00:00,29,17
2016-03-31 23:15:00,29,18


In [137]:

# zeros in the data

(resampled_data['total_pickups'] == 0).sum()

np.int64(3668)

In [138]:
epsilon_val = 10

resampled_data.replace({'total_pickups': {0 : epsilon_val}}, inplace=True)

In [140]:

(resampled_data['total_pickups'] == 0).sum()

np.int64(0)

In [141]:

from sklearn.metrics import mean_absolute_percentage_error

In [142]:
window_values = list(range(3,11,1))
window_values

[3, 4, 5, 6, 7, 8, 9, 10]

In [143]:
def calculate_best_window_value(windows):
    for window in windows:
        ind = window - 1
        y_pred = resampled_data['total_pickups'].rolling(window=window).mean().values[ind:]
        y = resampled_data['total_pickups'].values[ind:]
        error = mean_absolute_percentage_error(y, y_pred)
        print(f"For window value {window}, the MAPE is {error:.2f}")

In [144]:

calculate_best_window_value(window_values)

For window value 3, the MAPE is 0.20
For window value 4, the MAPE is 0.24
For window value 5, the MAPE is 0.28
For window value 6, the MAPE is 0.31
For window value 7, the MAPE is 0.35
For window value 8, the MAPE is 0.39
For window value 9, the MAPE is 0.42
For window value 10, the MAPE is 0.46


In [145]:
resampled_data['total_pickups'].ewm(alpha=0.9).mean()

tpep_pickup_datetime
2016-01-01 00:00:00     58.000000
2016-01-01 00:15:00    114.363636
2016-01-01 00:30:00    145.567568
2016-01-01 00:45:00    158.558056
2016-01-01 01:00:00    184.156062
                          ...    
2016-03-31 22:45:00     14.720768
2016-03-31 23:00:00     16.772077
2016-03-31 23:15:00     17.877208
2016-03-31 23:30:00     13.487721
2016-03-31 23:45:00     13.948772
Name: total_pickups, Length: 262080, dtype: float64

In [146]:

smoothing_values = np.arange(0.2,1,0.1)
smoothing_values

array([0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9])

In [147]:

def calculate_best_smoothing_value(values):
    y = resampled_data['total_pickups'].values
    for value in values:
        y_pred = resampled_data['total_pickups'].ewm(alpha=value).mean()
        error = mean_absolute_percentage_error(y, y_pred)
        print(f"For smoothing value {value:.1f}, the MAPE is {error:.2f}")

In [148]:

calculate_best_smoothing_value(smoothing_values)

For smoothing value 0.2, the MAPE is 0.41
For smoothing value 0.3, the MAPE is 0.27
For smoothing value 0.4, the MAPE is 0.20
For smoothing value 0.5, the MAPE is 0.16
For smoothing value 0.6, the MAPE is 0.12
For smoothing value 0.7, the MAPE is 0.09
For smoothing value 0.8, the MAPE is 0.06
For smoothing value 0.9, the MAPE is 0.03


In [149]:

# dataset with pickup smoothing applied (shifted by 1 to avoid data leakage)

resampled_data["avg_pickups"] = resampled_data['total_pickups'].ewm(alpha=0.4).mean().shift(1).round()

resampled_data

,region,total_pickups,avg_pickups
tpep_pickup_datetime,,,
2016-01-01 00:00:00,0,58,NaN
2016-01-01 00:15:00,0,120,58.0
2016-01-01 00:30:00,0,149,97.0
2016-01-01 00:45:00,0,160,123.0
2016-01-01 01:00:00,0,187,140.0
...,...,...,...
2016-03-31 22:45:00,29,14,17.0
2016-03-31 23:00:00,29,17,16.0
2016-03-31 23:15:00,29,18,16.0


In [150]:

# save the resampled data

resampled_data_save_path = "/kaggle/working/data/interim/final_data.csv"

resampled_data.to_csv(resampled_data_save_path, index=True)

## Model Building

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
# load the data

data_path = "/kaggle/working/data/interim/final_data.csv"

df = pd.read_csv(data_path, parse_dates=["tpep_pickup_datetime"])

In [2]:
# shape of the data

df.shape

(262080, 4)

In [3]:

# extract the day of week information
df["day_of_week"] = df["tpep_pickup_datetime"].dt.day_of_week

# extract the month information
df["month"] = df["tpep_pickup_datetime"].dt.month

In [4]:
# set the datetime column as index

df.set_index("tpep_pickup_datetime", inplace=True)
df

,region,total_pickups,avg_pickups,day_of_week,month
tpep_pickup_datetime,,,,,
2016-01-01 00:00:00,0,58,NaN,4,1
2016-01-01 00:15:00,0,120,58.0,4,1
2016-01-01 00:30:00,0,149,97.0,4,1
2016-01-01 00:45:00,0,160,123.0,4,1
2016-01-01 01:00:00,0,187,140.0,4,1
...,...,...,...,...,...
2016-03-31 22:45:00,29,14,17.0,3,3
2016-03-31 23:00:00,29,17,16.0,3,3
2016-03-31 23:15:00,29,18,16.0,3,3


In [5]:
# create the region grouper

region_grp = df.groupby("region")

region_grp.

In [6]:
# shifting periods

periods = list(range(1,5))

periods

[1, 2, 3, 4]

In [7]:
# generate the lag features

lag_features = region_grp["total_pickups"].shift(periods)

lag_features

,total_pickups_1,total_pickups_2,total_pickups_3,total_pickups_4
tpep_pickup_datetime,,,,
2016-01-01 00:00:00,NaN,NaN,NaN,NaN
2016-01-01 00:15:00,58.0,NaN,NaN,NaN
2016-01-01 00:30:00,120.0,58.0,NaN,NaN
2016-01-01 00:45:00,149.0,120.0,58.0,NaN
2016-01-01 01:00:00,160.0,149.0,120.0,58.0
...,...,...,...,...
2016-03-31 22:45:00,22.0,14.0,15.0,13.0
2016-03-31 23:00:00,14.0,22.0,14.0,15.0
2016-03-31 23:15:00,17.0,14.0,22.0,14.0


In [8]:
# merge them with the original df

data = pd.concat([lag_features,df],axis=1)

data

,total_pickups_1,total_pickups_2,total_pickups_3,total_pickups_4,region,total_pickups,avg_pickups,day_of_week,month
tpep_pickup_datetime,,,,,,,,,
2016-01-01 00:00:00,NaN,NaN,NaN,NaN,0,58,NaN,4,1
2016-01-01 00:15:00,58.0,NaN,NaN,NaN,0,120,58.0,4,1
2016-01-01 00:30:00,120.0,58.0,NaN,NaN,0,149,97.0,4,1
2016-01-01 00:45:00,149.0,120.0,58.0,NaN,0,160,123.0,4,1
2016-01-01 01:00:00,160.0,149.0,120.0,58.0,0,187,140.0,4,1
...,...,...,...,...,...,...,...,...,...
2016-03-31 22:45:00,22.0,14.0,15.0,13.0,29,14,17.0,3,3
2016-03-31 23:00:00,14.0,22.0,14.0,15.0,29,17,16.0,3,3
2016-03-31 23:15:00,17.0,14.0,22.0,14.0,29,18,16.0,3,3


In [9]:
print("The shape of the df before merger ", df.shape)
print("The shape of the df after merger ", data.shape)


The shape of the df before merger  (262080, 5)
The shape of the df after merger  (262080, 9)


In [10]:
# rows having missing values

data.isna().any(axis=1).sum()


np.int64(120)

In [11]:
# drop the missing values

data.dropna(inplace=True)
data.isna().any(axis=1).sum()

np.int64(0)

In [12]:
mapper = {name:f"lag_{ind+1}" for ind, name in enumerate(data.columns[0:4])}

mapper

{'total_pickups_1': 'lag_1',
 'total_pickups_2': 'lag_2',
 'total_pickups_3': 'lag_3',
 'total_pickups_4': 'lag_4'}

In [13]:
# replace the column names

data = data.rename(columns=mapper)

In [14]:
# number of rows in each month

data['month'].value_counts()

month
3    89280
1    89160
2    83520
Name: count, dtype: int64

In [15]:
data.loc[data["month"].isin([1,2]),"lag_1":"day_of_week"]

,lag_1,lag_2,lag_3,lag_4,region,total_pickups,avg_pickups,day_of_week
tpep_pickup_datetime,,,,,,,,
2016-01-01 01:00:00,160.0,149.0,120.0,58.0,0,187,140.0,4
2016-01-01 01:15:00,187.0,160.0,149.0,120.0,0,194,161.0,4
2016-01-01 01:30:00,194.0,187.0,160.0,149.0,0,180,175.0,4
2016-01-01 01:45:00,180.0,194.0,187.0,160.0,0,197,177.0,4
2016-01-01 02:00:00,197.0,180.0,194.0,187.0,0,185,185.0,4
...,...,...,...,...,...,...,...,...
2016-02-29 22:45:00,15.0,9.0,11.0,11.0,29,12,12.0,0
2016-02-29 23:00:00,12.0,15.0,9.0,11.0,29,17,12.0,0
2016-02-29 23:15:00,17.0,12.0,15.0,9.0,29,15,14.0,0


In [16]:
# split the data

trainset = data.loc[data["month"].isin([1,2]),"lag_1":"day_of_week"]

testset = data.loc[data["month"].isin([3]),"lag_1":"day_of_week"]
trainset 

,lag_1,lag_2,lag_3,lag_4,region,total_pickups,avg_pickups,day_of_week
tpep_pickup_datetime,,,,,,,,
2016-01-01 01:00:00,160.0,149.0,120.0,58.0,0,187,140.0,4
2016-01-01 01:15:00,187.0,160.0,149.0,120.0,0,194,161.0,4
2016-01-01 01:30:00,194.0,187.0,160.0,149.0,0,180,175.0,4
2016-01-01 01:45:00,180.0,194.0,187.0,160.0,0,197,177.0,4
2016-01-01 02:00:00,197.0,180.0,194.0,187.0,0,185,185.0,4
...,...,...,...,...,...,...,...,...
2016-02-29 22:45:00,15.0,9.0,11.0,11.0,29,12,12.0,0
2016-02-29 23:00:00,12.0,15.0,9.0,11.0,29,17,12.0,0
2016-02-29 23:15:00,17.0,12.0,15.0,9.0,29,15,14.0,0


In [21]:
# save the train and test data

train_data_save_path = "/kaggle/working/data/interim/train.csv"

test_data_save_path = "/kaggle/working/data/interim/test.csv"

trainset.to_csv(train_data_save_path, index=True)
testset.to_csv(test_data_save_path, index=True)

In [22]:
# make X_train and y_train

X_train = trainset.drop(columns=["total_pickups"])

y_train = trainset["total_pickups"]

In [23]:
X_train.shape

(172680, 7)

In [25]:
y_train.shape

(172680,)

In [30]:
# make X_test and y_test

X_test = testset.drop(columns=["total_pickups"])

y_test = testset["total_pickups"]

In [26]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_percentage_error

In [27]:
from sklearn import set_config

set_config(transform_output="pandas")

In [28]:

# encode the data

encoder = ColumnTransformer([
    ("ohe", OneHotEncoder(drop="first",sparse_output=False), ["region","day_of_week"])
], remainder="passthrough", n_jobs=-1,force_int_remainder_cols=False)

In [31]:
# encode the train and test data

X_train_encoded = encoder.fit_transform(X_train)
X_test_encoded = encoder.transform(X_test)

In [32]:
# encode the train and test data

X_train_encoded = encoder.fit_transform(X_train)
X_test_encoded = encoder.transform(X_test)

In [33]:
# train the model

lr = LinearRegression()

# fit on the training data
lr.fit(X_train_encoded, y_train)

LinearRegression()

In [37]:
# make predictions on the train data

y_pred_train = lr.predict(X_train_encoded)
# make predictions on the test data

y_pred_test = lr.predict(X_test_encoded)

In [38]:
# evaluate the baseline model

train_mape = mean_absolute_percentage_error(y_train, y_pred_train)

test_mape = mean_absolute_percentage_error(y_test, y_pred_test)

In [39]:
test_mape

0.30698710068880253

In [40]:
print(f"MAPE on trainset is {(train_mape * 100):.2f}%")
print(f"MAPE on testset is {(test_mape * 100):.2f}%")

MAPE on trainset is 31.75%
MAPE on testset is 30.70%


## Hyper Parmeter tuning

In [2]:
!pip install mlflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.2/44.2 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 62.1 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 72.0 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 39.3 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.4/228.4 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.5/136.5 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 25.7 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.0/216.0 kB 10.3 MB/s eta 0:00:00


In [3]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.svm import SVR

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor

import mlflow

In [4]:
mlflow.set_tracking_uri("https://dagshub.com/iamdebasishdas123/Uber-Demand-Prediction.mlflow")

In [7]:
!pip install dagshub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 274.0/274.0 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 4.5 MB/s eta 0:00:00
  Attempting uninstall: dacite
    Found existing installation: dacite 1.9.2
    Uninstalling dacite-1.9.2:
      Successfully uninstalled dacite-1.9.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydata-profiling 4.18.4 requires dacite<2,>=1.9, but you have dacite 1.6.0 which is incompatible.


In [8]:
import dagshub
dagshub.init(repo_owner='iamdebasishdas123', repo_name='Uber-Demand-Prediction', mlflow=True)

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=1c2052ca-96c0-4670-9792-d1f0ae52e197&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=2a46cac39779d8c298776ddbb359eac8fb7db29ffe3447ec52c20465793e31fc




Accessing as iamdebasishdas123

Initialized MLflow to track repo "iamdebasishdas123/Uber-Demand-Prediction"

Repository iamdebasishdas123/Uber-Demand-Prediction initialized!

In [10]:
# load the training and test data

train_data_path = "/kaggle/working/data/interim/train.csv"

test_data_path = "/kaggle/working/data/interim/test.csv"

train_df = pd.read_csv(train_data_path, parse_dates=["tpep_pickup_datetime"]).set_index("tpep_pickup_datetime")

test_df = pd.read_csv(test_data_path, parse_dates=["tpep_pickup_datetime"]).set_index("tpep_pickup_datetime")

train_df

,lag_1,lag_2,lag_3,lag_4,region,total_pickups,avg_pickups,day_of_week
tpep_pickup_datetime,,,,,,,,
2016-01-01 01:00:00,160.0,149.0,120.0,58.0,0,187,140.0,4
2016-01-01 01:15:00,187.0,160.0,149.0,120.0,0,194,161.0,4
2016-01-01 01:30:00,194.0,187.0,160.0,149.0,0,180,175.0,4
2016-01-01 01:45:00,180.0,194.0,187.0,160.0,0,197,177.0,4
2016-01-01 02:00:00,197.0,180.0,194.0,187.0,0,185,185.0,4
...,...,...,...,...,...,...,...,...
2016-02-29 22:45:00,15.0,9.0,11.0,11.0,29,12,12.0,0
2016-02-29 23:00:00,12.0,15.0,9.0,11.0,29,17,12.0,0
2016-02-29 23:15:00,17.0,12.0,15.0,9.0,29,15,14.0,0


In [11]:
# missing value in training data

train_df.isna().sum()

lag_1            0
lag_2            0
lag_3            0
lag_4            0
region           0
total_pickups    0
avg_pickups      0
day_of_week      0
dtype: int64

In [12]:
# missing values in the test data

test_df.isna().sum()

lag_1            0
lag_2            0
lag_3            0
lag_4            0
region           0
total_pickups    0
avg_pickups      0
day_of_week      0
dtype: int64

In [13]:
# make X_train and y_train

X_train = train_df.drop(columns=["total_pickups"])

y_train = train_df["total_pickups"]

In [15]:

X_test.head()

,lag_1,lag_2,lag_3,lag_4,region,avg_pickups,day_of_week
tpep_pickup_datetime,,,,,,,
2016-03-01 00:00:00,36.0,44.0,31.0,29.0,0,38.0,1
2016-03-01 00:15:00,41.0,36.0,44.0,31.0,0,39.0,1
2016-03-01 00:30:00,35.0,41.0,36.0,44.0,0,37.0,1
2016-03-01 00:45:00,47.0,35.0,41.0,36.0,0,41.0,1
2016-03-01 01:00:00,34.0,47.0,35.0,41.0,0,38.0,1


In [16]:
from sklearn import set_config

set_config(transform_output="pandas")

# encode the data

encoder = ColumnTransformer([
    ("ohe", OneHotEncoder(drop="first",sparse_output=False), ["region","day_of_week"])
], remainder="passthrough", n_jobs=-1,force_int_remainder_cols=False)

In [17]:
encoder

ColumnTransformer(force_int_remainder_cols=False, n_jobs=-1,
                  remainder='passthrough',
                  transformers=[('ohe',
                                 OneHotEncoder(drop='first',
                                               sparse_output=False),
                                 ['region', 'day_of_week'])])

In [18]:
# encode the train and test data

X_train_encoded = encoder.fit_transform(X_train)
X_test_encoded = encoder.transform(X_test)

In [19]:
import optuna
import tqdm 

In [20]:
# set the experiment

mlflow.set_experiment("Model Selection")

2026/09/07 17:17:10 INFO mlflow.tracking.fluent: Experiment with name 'Model Selection' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/633924a0084f421b8ee9308ded20e23c', creation_time=1788801430355, effective_trace_archival_retention=None, experiment_id='0', last_update_time=1788801430355, lifecycle_stage='active', name='Model Selection', tags={}, trace_location=None, workspace='default'>

*OPtuna*
Optuna works by framing hyperparameter optimization as an iterative search problem where a central orchestrator decides which parameter combinations to test based on past performance.

To build an Optuna optimization script, you only need four core components:

- An Objective Function: A standard Python function that accepts an Optuna trial object as its input.
It must contain your data loading, model initialization, training loop, and evaluation step.
- Suggested Hyperparameters: Inside your objective function, you must replace your fixed numbers with Optuna suggestion methods.Examples: trial.suggest_float(), trial.suggest_int(), or trial.suggest_categorical().
- A Return Metric: Your objective function must return a single scalar value (a float) at the very end.This is the metric Optuna will look at to judge whether the trial was a success or a failure.
- A Study Object: Generated via optuna.create_study().You must explicitly tell it whether to maximize your return metric (e.g., accuracy, F1-score) or minimize it (e.g., MSE loss, cross-entropy).

In [21]:
def objective(trial):
    # start the child run
    with mlflow.start_run(nested=True) as child:
        
        # model name search space
        list_of_models = ["LR", "RF", "GBR", "XGBR"]
        model_name = trial.suggest_categorical("model_name", list_of_models)
    
        if model_name == "LR":
            model = LinearRegression()
    
        elif model_name == "RF":
            n_estimators_rf = trial.suggest_int("n_estimators_rf",10,100,step=10)
            max_depth_rf = trial.suggest_int("max_depth_rf",3,10)
            model = RandomForestRegressor(n_estimators=n_estimators_rf, 
                                          max_depth=max_depth_rf, 
                                          random_state=42, n_jobs=-1)
    
        elif model_name == "GBR":
            n_estimators_gb = trial.suggest_int("n_estimators_gb",10,100,step=10)
            learning_rate_gb = trial.suggest_float("learning_rate_gb",1e-4,1e-1, log=True)
            model = GradientBoostingRegressor(n_estimators=n_estimators_gb, 
                                              learning_rate=learning_rate_gb,
                                             random_state=42)
    
        elif model_name == "XGBR":
            n_estimators_xgb = trial.suggest_int("n_estimators_xgb",10,100,step=10)
            learning_rate_xgb = trial.suggest_float("learning_rate_xgb",1e-4,1e-1, log=True)
            max_depth_xgb = trial.suggest_int("max_depth_xgb",3,10)
            model = XGBRegressor(n_estimators=n_estimators_xgb,
                                learning_rate=learning_rate_xgb,
                                max_depth=max_depth_xgb)
    
        # log the model name
        mlflow.log_param("model_name",model_name)
        
        # log the model parameters
        mlflow.log_params(model.get_params())
        
        # fit on the data
        model.fit(X_train_encoded,y_train)
    
        # get the predictions
        y_pred = model.predict(X_test_encoded)
    
        # calculate the loss
        loss = mean_absolute_percentage_error(y_test, y_pred)
    
        # log the metric
        mlflow.log_metric("MAPE",loss)
        return loss

In [22]:
# optimize the objective function

with mlflow.start_run(run_name="best_model", nested=True) as parent:

    # create a study object
    study = optuna.create_study(study_name="model_selection", direction="minimize")
    # optimize the objective function
    study.optimize(func=objective, n_trials=50, n_jobs=-1)
    
    # log the best parameters
    mlflow.log_params(study.best_params)
    # log the best error value
    mlflow.log_metric("Best_MAPE", study.best_value)

[I 2026-09-07 17:17:46,799] A new study created in memory with name: model_selection
[I 2026-09-07 17:18:10,655] Trial 1 finished with value: 6.087102890014648 and parameters: {'model_name': 'XGBR', 'n_estimators_xgb': 70, 'learning_rate_xgb': 0.0011387586709509074, 'max_depth_xgb': 5}. Best is trial 1 with value: 6.087102890014648.


🏃 View run fearless-kite-372 at: https://dagshub.com/iamdebasishdas123/Uber-Demand-Prediction.mlflow/#/experiments/0/runs/969e6cd6d1374c6781eb79912a95933a
🧪 View experiment at: https://dagshub.com/iamdebasishdas123/Uber-Demand-Prediction.mlflow/#/experiments/0


[I 2026-09-07 17:18:17,455] Trial 3 finished with value: 0.3252742186859445 and parameters: {'model_name': 'RF', 'n_estimators_rf': 100, 'max_depth_rf': 5}. Best is trial 3 with value: 0.3252742186859445.


🏃 View run beautiful-stoat-152 at: https://dagshub.com/iamdebasishdas123/Uber-Demand-Prediction.mlflow/#/experiments/0/runs/dfc66fe58cfd4bc287beb70d7a4cf6a4
🧪 View experiment at: https://dagshub.com/iamdebasishdas123/Uber-Demand-Prediction.mlflow/#/experiments/0


[I 2026-09-07 17:18:28,239] Trial 4 finished with value: 6.446369647979736 and parameters: {'model_name': 'XGBR', 'n_estimators_xgb': 70, 'learning_rate_xgb': 0.0002952632025340156, 'max_depth_xgb': 6}. Best is trial 3 with value: 0.3252742186859445.


...

🏃 View run marvelous-sloth-992 at: https://dagshub.com/iamdebasishdas123/Uber-Demand-Prediction.mlflow/#/experiments/0/runs/558490d7cb684cefae9edde9720b0650
🧪 View experiment at: https://dagshub.com/iamdebasishdas123/Uber-Demand-Prediction.mlflow/#/experiments/0
🏃 View run resilient-toad-715 at: https://dagshub.com/iamdebasishdas123/Uber-Demand-Prediction.mlflow/#/experiments/0/runs/fbb692e146ca49358f0f9e3747ce0187
🧪 View experiment at: https://dagshub.com/iamdebasishdas123/Uber-Demand-Prediction.mlflow/#/experiments/0


[I 2026-09-07 17:25:06,769] Trial 45 finished with value: 0.2896069414536356 and parameters: {'model_name': 'RF', 'n_estimators_rf': 20, 'max_depth_rf': 10}. Best is trial 28 with value: 0.2896069414536355.
[I 2026-09-07 17:25:07,105] Trial 47 finished with value: 0.2896069414536356 and parameters: {'model_name': 'RF', 'n_estimators_rf': 20, 'max_depth_rf': 10}. Best is trial 28 with value: 0.2896069414536355.


🏃 View run monumental-trout-153 at: https://dagshub.com/iamdebasishdas123/Uber-Demand-Prediction.mlflow/#/experiments/0/runs/e9eaadf58ee14ed59b59e37c6eb7d3e8
🧪 View experiment at: https://dagshub.com/iamdebasishdas123/Uber-Demand-Prediction.mlflow/#/experiments/0
🏃 View run ambitious-quail-790 at: https://dagshub.com/iamdebasishdas123/Uber-Demand-Prediction.mlflow/#/experiments/0/runs/703167fefb8746f191dcb88c270c0471
🧪 View experiment at: https://dagshub.com/iamdebasishdas123/Uber-Demand-Prediction.mlflow/#/experiments/0


[I 2026-09-07 17:25:18,775] Trial 48 finished with value: 0.2896069414536355 and parameters: {'model_name': 'RF', 'n_estimators_rf': 20, 'max_depth_rf': 10}. Best is trial 28 with value: 0.2896069414536355.
[I 2026-09-07 17:25:19,131] Trial 46 finished with value: 0.2896069414536356 and parameters: {'model_name': 'RF', 'n_estimators_rf': 20, 'max_depth_rf': 10}. Best is trial 28 with value: 0.2896069414536355.
[I 2026-09-07 17:25:27,909] Trial 49 finished with value: 0.29259642609457387 and parameters: {'model_name': 'RF', 'n_estimators_rf': 20, 'max_depth_rf': 9}. Best is trial 28 with value: 0.2896069414536355.


🏃 View run shivering-deer-320 at: https://dagshub.com/iamdebasishdas123/Uber-Demand-Prediction.mlflow/#/experiments/0/runs/4211401b16ee42479d026386eb2f7a33
🧪 View experiment at: https://dagshub.com/iamdebasishdas123/Uber-Demand-Prediction.mlflow/#/experiments/0
🏃 View run best_model at: https://dagshub.com/iamdebasishdas123/Uber-Demand-Prediction.mlflow/#/experiments/0/runs/43a17cb9748848e293386e0691674fbc
🧪 View experiment at: https://dagshub.com/iamdebasishdas123/Uber-Demand-Prediction.mlflow/#/experiments/0


In [23]:
# best value

study.best_value

0.2896069414536355

In [24]:
# best parameters

study.best_params

{'model_name': 'RF', 'n_estimators_rf': 20, 'max_depth_rf': 10}

In [25]:
# model value counts

study.trials_dataframe()['params_model_name'].value_counts()

params_model_name
RF      26
LR      10
GBR      8
XGBR     6
Name: count, dtype: int64